In [19]:
import os
import requests

In [20]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response= requests.get(url, timeout=30)
    response.raise_for_status()
    print(response.content[:50])
    with open(file_path, "wb") as f:
        f.write(response.content)

In [21]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [22]:
import re

In [23]:
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
result

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

In [24]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
preprocessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

In [25]:
len(preprocessed)

4690

2.3 Converting tokens into token IDs

In [26]:
all_words = sorted(set(preprocessed))
all_words[:10]

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';']

In [27]:
vocab_size = len(all_words)
vocab_size

1130

In [28]:
vocab = {token: integer for integer, token in enumerate(all_words)}
vocab.__len__()

1130

In [29]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [30]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        # replace incorrect formatting of punctuations ("hello , there" -> "hello, there")
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [31]:
tokenizer = SimpleTokenizerV1(vocab)

In [32]:
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

In [33]:
ids = tokenizer.encode(text)
ids[:5]

[1, 56, 2, 850, 988]

In [34]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [35]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

2.4 Adding special context tokens

In [36]:
tokenizer = SimpleTokenizerV1(vocab)
text = "Hello, do you like tea. Is this-- a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [38]:
all_tokens = sorted(list(set(preprocessed)))
len(all_tokens)
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}
len(vocab)

1132

In [39]:
for item in list(vocab.items())[-5:]:
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [40]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        # replace incorrect formatting of punctuations ("hello , there" -> "hello, there")
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [41]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

In [42]:
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [43]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [44]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

2.5 BytePair encoding

In [45]:
# tiktoken pip library used

In [46]:
import importlib
import tiktoken

In [47]:
print("tiktoken version: ", importlib.metadata.version("tiktoken"))
tiktoken.__version__

tiktoken version:  0.14.0


'0.14.0'

In [48]:
tokenizer = tiktoken.get_encoding("gpt2")

In [49]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

In [50]:
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
integers[:10]

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554]

In [51]:
strings = tokenizer.decode(integers)
strings

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

2.6 Data sampling with a sliding window

In [52]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [53]:
enc_text = tokenizer.encode(raw_text)

In [54]:
len(enc_text)

5145

In [55]:
enc_sample = enc_text[50:]

In [56]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

In [57]:
print(x)
y

[290, 4920, 2241, 287]


[4920, 2241, 287, 257]

In [58]:
# one-by-one prediction
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [59]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [60]:
import torch

In [61]:
torch.__version__

'2.14.0+cu130'

In [62]:
from torch.utils.data import Dataset, DataLoader

In [63]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize all text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length + 1"

        # sliding windows to chunk the book into sequences of len max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [64]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [65]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [82]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

In [83]:
second_batch = next(data_iter)
second_batch

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]

In [85]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
    )

In [102]:
data_iter = iter(dataloader)
data_iter

In [107]:
inputs, targets = next(data_iter)
inputs, targets

(tensor([[  286,   616,  4286,   705],
         [ 1014,   510,    26,   475],
         [  314,   836,   470,   892],
         [  286,   326,    11,  1770],
         [   13,  8759,  2763,   438],
         [ 1169,  2994,   284,   943],
         [17034,   318,   477,   314],
         [  892,   286,   526,   383]]),
 tensor([[  616,  4286,   705,  1014],
         [  510,    26,   475,   314],
         [  836,   470,   892,   286],
         [  326,    11,  1770,    13],
         [ 8759,  2763,   438,  1169],
         [ 2994,   284,   943, 17034],
         [  318,   477,   314,   892],
         [  286,   526,   383,  1573]]))

2.7 Creating token embeddings

In [108]:
input_ids = torch.tensor([2, 3, 5, 1])

In [110]:
vocab_size = 6
output_dim = 3

In [115]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [114]:
embedding_layer.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

In [117]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [119]:
embedding_layer(input_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

2.8 Encoding word positions